# 2. Testiranje modela

U 1. bilježnici smo naučili model i spremili ga u datoteku `eurosat_cnn.pt`. Sada ga provjeravamo na **testnom skupu**: 2700 slika koje model nikad nije vidio – ni tijekom treniranja ni pri odabiru najbolje epohe. Tek je to pošten ispit.

Osim ukupne točnosti gledamo:
- točnost za svaki razred posebno,
- **matricu zabune** – koje razrede model međusobno zamjenjuje,
- **test otpornosti** – radi li model i na iskvarenim slikama (šum, zamućenje, kompresija...).

## Korak 0: Učitavanje modela i podataka

In [ ]:
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader

import eurosat_common as ec

BATCH_SIZE = 128
NUM_WORKERS = 2  # ako dobiješ grešku "bus error" ili "shared memory", stavi 0

device = ec.get_device()
model = ec.load_model(ec.MODEL_PATH, device)
dataset = load_dataset(ec.DATASET_ID)


def make_loader(split_name):
    """DataLoader za jedan dio skupa podataka (bez augmentacije)."""
    return DataLoader(ec.EuroSATDataset(dataset[split_name], ec.eval_transform),
                      batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

## Korak 1: Ukupna točnost na testnom skupu

In [ ]:
test_loss, test_acc, preds, labels = ec.evaluate(model, make_loader("test"), device)

print(f"Gubitak na testnom skupu: {test_loss:.3f}")
print(f"Točnost na testnom skupu: {test_acc:.1%}  ({(preds == labels).sum().item()} od {len(labels)} slika točno)")

Usporedi ovo s točnošću na validaciji (ispisana je pri učitavanju modela). Ako su slične, model jednako dobro radi i na slikama koje nikad nije vidio.

## Korak 2: Točnost po razredima

Ukupna točnost može sakriti da model neki razred prepoznaje puno lošije od ostalih.

In [ ]:
per_class_acc = []
for class_index, name in enumerate(ec.CLASS_NAMES):
    is_this_class = labels == class_index
    accuracy = (preds[is_this_class] == class_index).float().mean().item()
    per_class_acc.append(accuracy)
    print(f"{name:<34} {accuracy:6.1%}")

plt.figure(figsize=(10, 4.5))
plt.barh(ec.CLASS_NAMES, [a * 100 for a in per_class_acc], color="#2a78d6")
for i, accuracy in enumerate(per_class_acc):
    plt.text(accuracy * 100 + 1, i, f"{accuracy:.1%}", va="center")
plt.gca().invert_yaxis()
plt.xlim(0, 112)
plt.xlabel("točnost (%)")
plt.title("Točnost po razredima na testnom skupu")
plt.tight_layout()
plt.show()

## Korak 3: Matrica zabune

**Redak** je pravi razred, a **stupac** razred koji je model predvidio. Brojevi na dijagonali su točni odgovori, a sve izvan dijagonale su greške.

Na primjer: broj u retku *rijeka* i stupcu *autocesta ili cesta* govori koliko je rijeka model zamijenio za cestu.

In [ ]:
n = ec.NUM_CLASSES
confusion = torch.bincount(labels * n + preds, minlength=n * n).reshape(n, n)

fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(confusion, cmap="Blues")
threshold = confusion.max().item() / 2
for true_class in range(n):
    for pred_class in range(n):
        count = confusion[true_class, pred_class].item()
        ax.text(pred_class, true_class, count, ha="center", va="center",
                color="white" if count > threshold else "black")
ax.set_xticks(range(n), ec.CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(n), ec.CLASS_NAMES)
ax.set_xlabel("predviđeni razred")
ax.set_ylabel("pravi razred")
ax.set_title("Matrica zabune")
plt.tight_layout()
plt.show()

In [ ]:
mistakes = confusion.clone()
mistakes.fill_diagonal_(0)  # dijagonala su točni odgovori - njih ne brojimo

print("Najčešće zamjene (pravi razred -> predviđeni razred):")
for index in mistakes.flatten().argsort(descending=True)[:5].tolist():
    true_class, pred_class = divmod(index, n)
    print(f"  {ec.CLASS_NAMES[true_class]} -> {ec.CLASS_NAMES[pred_class]}: {mistakes[true_class, pred_class].item()} puta")

**Razmisli:** zašto se baš ti razredi miješaju? Pašnjak, grmlje i usjevi su zelene površine bez jasne strukture. Autocesta i rijeka su obje duge, uske linije. I čovjeku bi neke od tih slika bile teške!

## Korak 4: Test otpornosti

Prave snimke nisu uvijek savršene: šum senzora, kompresija pri slanju sa satelita, zamućenje... Skup podataka ima 7 iskvarenih verzija istih 2700 testnih slika. Prvo pogledajmo kako izgledaju:

In [ ]:
index = 0  # promijeni broj da vidiš drugu sliku
split_names = ["test"] + list(ec.CORRUPTIONS)
titles = ["original"] + list(ec.CORRUPTIONS.values())

fig, axes = plt.subplots(1, len(split_names), figsize=(18, 2.8))
for ax, split_name, title in zip(axes, split_names, titles):
    ax.imshow(dataset[split_name][index]["image"])
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.show()

Sada isti model provjerimo na svakoj iskvarenoj verziji testnog skupa:

In [ ]:
corrupted_acc = {}
print(f"{'original':<20} točnost {test_acc:6.1%}")
for split_name, title in ec.CORRUPTIONS.items():
    _, accuracy, _, _ = ec.evaluate(model, make_loader(split_name), device)
    corrupted_acc[title] = accuracy
    print(f"{title:<20} točnost {accuracy:6.1%}   (pad {(test_acc - accuracy) * 100:4.1f} postotnih bodova)")

In [ ]:
plt.figure(figsize=(11, 4.5))
plt.bar(corrupted_acc.keys(), [a * 100 for a in corrupted_acc.values()], color="#2a78d6")
for i, accuracy in enumerate(corrupted_acc.values()):
    plt.text(i, accuracy * 100 + 1.5, f"{accuracy:.1%}", ha="center")
plt.axhline(test_acc * 100, color="gray", linestyle="--", label=f"originalne slike: {test_acc:.1%}")
plt.legend(loc="upper right")
plt.ylim(0, 122)
plt.ylabel("točnost (%)")
plt.title("Točnost na iskvarenim slikama")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

**Zašto neka iskvarenja škode više od drugih?** Konvolucijski slojevi prepoznaju razrede po sitnim detaljima i teksturama, a šum i mrlje prekrivaju upravo te detalje novim, lažnim uzorcima. Model je tijekom treniranja vidio samo čiste snimke (uz zrcaljenje i malu promjenu svjetline i kontrasta), pa ovakve slike nikad prije nije "sreo".

**Ideja za istraživanje:** dodaj u `train_transform` (u `eurosat_common.py`) i zamućenje, npr. `transforms.GaussianBlur(kernel_size=3)` prije `ToTensor()`. Ponovno treniraj model i provjeri je li otporniji. Je li se točnost na originalnim slikama promijenila?

## Sažetak

In [ ]:
worst = min(corrupted_acc, key=corrupted_acc.get)
print(f"Točnost na testnom skupu: {test_acc:.1%}")
print(f"Najteže iskvarenje: {worst} ({corrupted_acc[worst]:.1%})")
print("Sljedeći korak: bilježnica 3_koristenje.ipynb")